## Assignment 2

*100 points (7% of course grade)*</br>
*Assigned: Mon, Sep 29th*</br>
**Due: Tue, Oct 14th, 23:59**

This homework should be done in parts as soon as (< 1 week) relevant topics are covered in lectures. If you wait until the last minute, you might be overwhelmed.

You must turn in the required files electronically, including this Notebook (A2.ipynb). Please follow the submission instructions for each problem carefully.

In this assignment, you need to solve two problems. In Problem 1, you will write a few SQL queries to query the beers database (the same as in A1). In Problem 2, you will answer two questions in database design theory.

## Setup your PostgreSQL

You will need this setup to create a database on your machine and to test your queries. Please follow our [setup instructions](https://canvas.sfu.ca/courses/91482/pages/0-main-entrance-general-guidance-on-cmpt-354-environment-setup) on Canvas.

### Problem 1: Query with SQL (64%)

Consider again the beer drinker's database from Assignment 1 with following schema (key columns underlined).

- drinker (<u>name</u>, address)
- bar (<u>name</u>, address)
- beer (<u>name</u>, brewer)
- frequents (<u>drinker</u>, <u>bar</u>, times_a_week)
- likes (<u>drinker</u>, <u>beer</u>)
- serves (<u>bar</u>, <u>beer</u>, price)


#### **Preliminary**

After you finish the PostgreSQL setup, you will be able to run the PostgreSQL's interpreter in your command line tools by running `psql beers -U [your username]` or simply `psql beers` (If you are using the Docker container we provide, the username is by default `postgres`) in your terminal or command line. You may use either pgAdmin or the command line tool.

Same as in Assignment 1, you will need to type your SQL queries in the cells below. 


Now your homework question is to write SQL queries to answer following questions. One major difference between SQL and relational algebra queries is the *bag semantics*: you may need to use DISTINCT in your SELECT statement or aggregate functions to deduplicate results.

Please fill your answer in each cell (and **ONLY the query**) and **DO NOT add or remove** any cells to make the TAs' life easier in evaluating your queries. Questions (1)-(4) are worth 6 points each; (5)-(9) are worth 8 points each.


#### 0. (example) Find names of all bars that Eve frequents.

In [ ]:
/* input your answer in this cell: */

SELECT bar FROM frequents WHERE drinker = 'Eve';

#### 1. Find names of bars that serve either Amstel or Corona at price higher than \\$2.

In [ ]:
/* input your answer in this cell: */
SELECT DISTINCT bar
FROM serves
WHERE (beer = 'Amstel' OR beer = 'Corona')
  AND price > 2;

#### 2. Find the names of all drinkers that like Corona but frequent no bars that serve Corona.

In [ ]:
/* input your answer in this cell: */
SELECT DISTINCT drinker
FROM likes
WHERE beer = 'Corona'
EXCEPT
SELECT frequents.drinker
FROM frequents 
JOIN serves
ON frequents.bar = serves.bar
WHERE serves.beer = 'Corona'

#### 3. Find the names of all bars that serve at least 5 beers.

In [ ]:
/* input your answer in this cell: */
SELECT bar
FROM serves 
GROUP BY bar
HAVING COUNT(DISTINCT beer) >= 5

#### 4. Find the pair of drinkers who frequent bars the same total number of times per week. Don't list (drinkerA, drinkerA). Only list pairs (drinkerA, drinkerB) where drinkerA < drinkerB  in the answer; don't list (drinkerB, drinkerA) again.

In [ ]:
/* input your answer in this cell: */
/*need to find TOTAL of # times to ALL bars of EACH drinker*/

WITH total_times_a_week AS (
    SELECT SUM(f.times_a_week) sum_f, f.drinker
    FROM frequents AS f
    GROUP BY f.drinker) 

SELECT DISTINCT t1.drinker AS d1, t2.drinker AS d2
FROM total_times_a_week AS t1, total_times_a_week AS t2
WHERE t1.sum_f = t2.sum_f
AND t1.drinker < t2.drinker

#### 5. Find names of all drinkers who frequent *only* those bars that serve *some* beers they like (drinkers who frequent no bars are included).


In [ ]:
/* input your answer in this cell: */
SELECT name
FROM drinker

EXCEPT

SELECT f.drinker
FROM frequents f
WHERE f.bar NOT IN (
    SELECT s.bar
    FROM serves s
    JOIN likes l ON s.beer = l.beer
    WHERE l.drinker = f.drinker
);

#### 6. Find the name of each beer such that in every bar where it is served, it is liked by some drinker who frequents this bar (If a beer is served nowhere, it should be returned as well).

In [ ]:
/* input your answer in this cell: */
SELECT b.name
FROM beer b
WHERE NOT EXISTS (
    SELECT *
    FROM Serves s
    WHERE s.beer = b.name
      AND NOT EXISTS (
          SELECT *
          FROM Likes l
          JOIN Frequents f ON l.drinker = f.drinker
          WHERE l.beer = b.name
            AND f.bar = s.bar
      )
)


#### 7. Calculate some statistics for each bar: (1) total number of drinkers who frequent it, (2) average price of beers it serves, and (3) name of the drinker who frequents it the maximum number of times a week (the most regular customer!). i.e., your query should output total_num_drinker, avg_price, most_reg_drinker
In case of ties in (3), rank the drinkers by the alphabetical order or their names. Sort the output by the number of drinkers (in descending order), in case of ties,
- sort by bar in alphabetical order. You need to list every bar, even if it
is not frequented by anyone
- (show 0 as the total number of drinkers in this case and NULL as the most
regular customer) or
- serves no beers (show NULL as average price in this case).

In [ ]:
/* input your answer in this cell: */
/* note: need GROUP BY, otherwise its for whole table */
SELECT
  b.name,
  COALESCE(table1.total_num_drinker, 0),
  table2.avg_price,
  table3.most_reg_drinker
FROM bar b
  LEFT JOIN (
    SELECT
      table1.bar,
      COUNT(DISTINCT table1.drinker) AS total_num_drinker
    FROM frequents
    GROUP BY
      bar
  ) AS table1 ON b.name = table1.bar
  LEFT JOIN(
    (
      SELECT
        bar,
        AVG(price) AS avg_price
      FROM serves
      GROUP BY
        bar
    )
  ) AS table2 ON b.name = table2.bar
  LEFT JOIN(
    (
      SELECT
        f1.bar,
        f1.drinker AS most_reg_drinker
      FROM frequents f1
      WHERE f1.times_a_week = (
          SELECT MAX(f2.times_a_week)
          FROM frequents f2
          WHERE f2.bar = f1.bar
        )
      ORDER BY
        f1.drinker
    )
  ) AS table3 ON b.name = table3.bar
ORDER BY
  table1.total_num_drinker DESC,
  b.name ASC

#### 8. Find all (bar1, bar2) pairs where: (1) at least one same beer is served, and (2) the average number of visits per drinker (exclude those who do not frequent any bar) per week of bar1 is more than 50% higher than the average number of visits per drinker of bar2.


In [ ]:
/* input your answer in this cell: */
WITH BarPair AS (
    SELECT
      DISTINCT s1.bar AS bar1,
      s2.bar AS bar2
    FROM serves s1
      JOIN serves s2 ON s1.beer = s2.beer 
  ), AvgVisits AS (
    SELECT
      bar,
      AVG(times_a_week) AS avg_visit_of_bar
    FROM frequents
    GROUP BY bar
  )
SELECT bp.bar1, bp.bar2
FROM BarPair bp
JOIN AvgVisits a1 ON bp.bar1 = a1.bar 
JOIN AvgVisits a2 ON bp.bar2 = a2.bar 
WHERE a1.avg_visit_of_bar > 1.5*a2.avg_visit_of_bar 

#### 9. Find, for each beer, its lowest serving price and the bar(s) serving it at this price. The output should be a list of (beer, price, bar) triples. If some beer B is not served anywhere, you should still output (B, NULL, NULL).


In [ ]:
/* input your answer in this cell: */

SELECT b.name, lowestPrice.price, lowestPrice.bar
FROM beer b
LEFT JOIN (
	SELECT bar, beer, price FROM serves s1
	WHERE s1.price = (
		SELECT MIN (price)
		FROM serves s2
		WHERE s1.beer = s2.beer
	)
)  AS lowestPrice
ON lowestPrice.beer = b.name


## Problem 2: Database design theory (36% = 16% + 20%)

#### 1. Consider a relation R with five attributes ABCDE. You are given the following dependencies: A -> D; DE -> C; CB -> A. 
1. For each FD $X \rightarrow Y$, compute the closure $X^+$
2. List all keys in R.
3. Is R in 3NF? Explain.
4. Is R in BCNF? Explain.

`Write your answers for 1,2,3,4 in this cell`

1. 
$A^+$ = {A,D}

$DE^+$ = {D,E,C}

$CB^+$ = {C,B,A,D}

2. 
BEA

BED

BEC

3.
Yes

Because D,C,A are parts of key

4.
No

Because none of the LHSs are superkeys

#### 2. Consider the following table storing information about Pals, jobs, and production structures in a simplified version of Palworld: R(pid, sid, species, structure_name, work_speed, job_type, food_consumption) and a set of functional dependencies:
* FD1: species, structure_name -> work_speed
* FD2: sid -> structure_name, job_type
* FD3: pid -> species, food_consumption
* FD4: structure_name -> job_type

`Decompose the schema into BCNF by (1) filling out the steps below, and (2) writing the names of the
relations in the final solution. Note: You may not need all four steps in your decompositions.`

**Step 1**
- Initial table: `R(pid, sid, species, structure_name, work_speed, job_type, food_consumption)`
- Violating FD $X \rightarrow Y$: species, structure_name -> work_speed
- Schema of new table-1: species, structure_name, work_speed
- Schema of new table-2: pid, sid, species, structure_name, job_type, food_consumption

**Step 2**
- Initial table: pid, sid, species, structure_name, job_type, food_consumption
- Violating FD $X \rightarrow Y$: sid -> structure_name, job_type
- Schema of new table-1: sid, structure_name, job_type
- Schema of new table-2: pid, sid, species, food_consumption

**Step 3**
- Initial table: pid, sid, species, food_consumption
- Violating FD $X \rightarrow Y$: pid -> species, food_consumption
- Schema of new table-1: pid, species, food_consumption
- Schema of new table-2: pid, sid

**Step 4**
- Initial table: 
- Violating FD $X \rightarrow Y$:
- Schema of new table-1:
- Schema of new table-2:

**Names of relations in the final solution:**

species, structure_name, work_speed

sid, structure_name, job_type

pid, species, food_consumption

pid, sid

## Submission instruction

1. For problem 1, answer the questions (1)-(9) in the Markdown cells

2. For problem 2.1, answer questions 1,2,3,4 in the given cell.

2. For problem 2.2, fill the steps you need after the `:` and write down the names of relations in the final solution

3. Compress your A2.ipynb (this file) into A2.zip and submit on Canvas